<a href="https://colab.research.google.com/github/KiranAslam/PPE_Detection_system/blob/main/PPE_detection_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="WiZGNgO3LUcdN4N3d1MA")
project = rf.workspace("roboflow-universe-projects").project("safety-vests")
version = project.version(14)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 4.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 5.0.0.93
    Uninstalling opencv-python-headless-5.0.0.93:
      Successfully uninstalled opencv-python-headless-5.0.0.93
  Attempting uninstall: typer
    Found existing installation: typer 0.26.8
    Uninstalling typer-0.26.8:
      Successfully uninstalled typer-0.26.8
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Safety-Vests-14 in yolov8:: 100%|██████████| 29630/29630 [00:05<00:00, 5624.50it/s]


In [2]:
!pip install -q ultralytics
from ultralytics.utils.downloads import download
from pathlib import Path

# Agar ultralytics install nahi hai
!pip install -q ultralytics

# Construction-PPE dataset download karo
download(
    "https://github.com/ultralytics/assets/releases/download/v0.0.0/construction-ppe.zip",
    dir="datasets"
)

# Verify karo ke extract ho gaya
import os
CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"
print("Contents:", os.listdir(CONSTRUCTION_PPE_PATH))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Unzipping datasets/construction-ppe.zip to /content/datasets/construction-ppe...: 100% ━━━━━━━━━━━━ 2852/2852 2.1Kfiles/s 1.3s
Contents: ['data.yaml', 'images', 'LICENSE', 'labels']


In [3]:
import yaml
import os
from pathlib import Path
from collections import Counter

# ===== Roboflow dataset ka path =====
DATASET_PATH = dataset.location   # agar variable already available hai to yeh use ho jayega
print("Dataset path:", DATASET_PATH)

# ===== Step 1: Classes dekho =====
yaml_path = os.path.join(DATASET_PATH, "data.yaml")
with open(yaml_path, "r") as f:
    data_yaml = yaml.safe_load(f)

print("\nClasses:")
for idx, name in enumerate(data_yaml['names']):
    print(f"  {idx}: {name}")

# ===== Step 2: "No Vest" class ID automatically dhundo =====
NO_VEST_ID = None
for idx, name in enumerate(data_yaml['names']):
    if "no" in name.lower() and "vest" in name.lower():
        NO_VEST_ID = idx
        break

if NO_VEST_ID is None:
    print("\n⚠️ 'No Vest' class nahi mili — upar classes list dekh ke bata dena exact naam kya hai")
else:
    print(f"\n'No Vest' class mil gayi -> ID: {NO_VEST_ID} ({data_yaml['names'][NO_VEST_ID]})")

# ===== Step 3: Har split mein count nikalo =====
splits = ["train", "valid", "test"]
total_no_vest_images = 0

for split in splits:
    lbl_dir = os.path.join(DATASET_PATH, split, "labels")
    img_dir = os.path.join(DATASET_PATH, split, "images")

    if not os.path.exists(lbl_dir):
        print(f"{split}: labels folder nahi mila")
        continue

    images_with_no_vest = []
    total_no_vest_boxes = 0

    for txt_file in Path(lbl_dir).glob("*.txt"):
        has_no_vest = False
        with open(txt_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts and int(parts[0]) == NO_VEST_ID:
                    has_no_vest = True
                    total_no_vest_boxes += 1
        if has_no_vest:
            images_with_no_vest.append(txt_file.stem)

    total_no_vest_images += len(images_with_no_vest)
    print(f"\n{split}:")
    print(f"  Total images with 'no-vest' box: {len(images_with_no_vest)}")
    print(f"  Total 'no-vest' boxes: {total_no_vest_boxes}")

print(f"\n===== GRAND TOTAL: {total_no_vest_images} images with no-vest across all splits =====")

Dataset path: /content/Safety-Vests-14

Classes:
  0: no_safety_vest
  1: safety_vest

'No Vest' class mil gayi -> ID: 0 (no_safety_vest)

train:
  Total images with 'no-vest' box: 3390
  Total 'no-vest' boxes: 7167

valid:
  Total images with 'no-vest' box: 189
  Total 'no-vest' boxes: 361

test:
  Total images with 'no-vest' box: 108
  Total 'no-vest' boxes: 226

===== GRAND TOTAL: 3687 images with no-vest across all splits =====


In [4]:
from pathlib import Path
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
    return counter

names = ["helmet","gloves","vest","boots","goggles","none","Person",
         "no_helmet","no_goggle","no_gloves","no_boots"]

train_counts = count_classes("datasets/construction-ppe/labels/train", names)


datasets/construction-ppe/labels/train:
  0 (helmet): 1357 boxes
  1 (gloves): 1162 boxes
  2 (vest): 1283 boxes
  3 (boots): 1251 boxes
  4 (goggles): 427 boxes
  5 (none): 654 boxes
  6 (Person): 1790 boxes
  7 (no_helmet): 400 boxes
  8 (no_goggle): 337 boxes
  9 (no_gloves): 442 boxes
  10 (no_boots): 88 boxes


In [5]:
import os
from pathlib import Path

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

# OLD ID -> NEW ID (boots=3 aur no_boots=10 hata diya, baaki sab rakha)
ID_REMAP = {
    0: 0,   # helmet
    1: 1,   # gloves
    2: 2,   # vest
    4: 3,   # goggles
    5: 4,   # none
    6: 5,   # Person
    7: 6,   # no_helmet
    8: 7,   # no_goggle
    9: 8,   # no_gloves
    # 3 (boots), 10 (no_boots) -> DROP
}

def remap_labels(label_dir):
    processed = 0
    for txt_file in Path(label_dir).glob("*.txt"):
        new_lines = []
        with open(txt_file, "r") as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                old_id = int(parts[0])
                if old_id in ID_REMAP:
                    parts[0] = str(ID_REMAP[old_id])
                    new_lines.append(" ".join(parts))
        with open(txt_file, "w") as f:
            if new_lines:
                f.write("\n".join(new_lines) + "\n")
        processed += 1
    print(f"{label_dir}: {processed} files remapped")

for split in ["train", "val", "test"]:
    remap_labels(f"{CONSTRUCTION_PPE_PATH}/labels/{split}")

print("\n✅ Boots aur no_boots hata diye, IDs remap ho gayi")

datasets/construction-ppe/labels/train: 1142 files remapped
datasets/construction-ppe/labels/val: 143 files remapped
datasets/construction-ppe/labels/test: 141 files remapped

✅ Boots aur no_boots hata diye, IDs remap ho gayi


In [6]:
import yaml

with open("/content/Safety-Vests-14/data.yaml", "r") as f:
    d = yaml.safe_load(f)

print("Safety-Vests-14 classes:")
for idx, name in enumerate(d['names']):
    print(f"  {idx}: {name}")

Safety-Vests-14 classes:
  0: no_safety_vest
  1: safety_vest


In [7]:
import os, shutil, random
from pathlib import Path

random.seed(42)

SAFETY_VEST_PATH = "/content/Safety-Vests-14"
NO_VEST_SOURCE_ID = 0       # ✅ confirmed
NO_VEST_TARGET_ID = 9       # naya class ID

TARGET_COUNTS = {
    "train": 960,
    "valid": 120,
    "test": 120,
}
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

total_added = 0

for split, target_n in TARGET_COUNTS.items():
    src_lbl_dir = os.path.join(SAFETY_VEST_PATH, split, "labels")
    src_img_dir = os.path.join(SAFETY_VEST_PATH, split, "images")

    dst_split = SPLIT_MAP[split]
    dst_img_dir = os.path.join(CONSTRUCTION_PPE_PATH, "images", dst_split)
    dst_lbl_dir = os.path.join(CONSTRUCTION_PPE_PATH, "labels", dst_split)
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    candidates = []
    for txt_file in Path(src_lbl_dir).glob("*.txt"):
        with open(txt_file) as f:
            lines = f.readlines()
        has_no_vest = any(line.split() and int(line.split()[0]) == NO_VEST_SOURCE_ID for line in lines)
        if has_no_vest:
            candidates.append(txt_file)

    take_n = min(target_n, len(candidates))
    if take_n < target_n:
        print(f"⚠️ {split}: sirf {len(candidates)} available hain, {target_n} nahi mil sakti")

    selected = random.sample(candidates, take_n)

    added = 0
    for txt_file in selected:
        new_lines = []
        with open(txt_file) as f:
            for line in f:
                parts = line.strip().split()
                if parts and int(parts[0]) == NO_VEST_SOURCE_ID:
                    parts[0] = str(NO_VEST_TARGET_ID)
                    new_lines.append(" ".join(parts))

        img_stem = txt_file.stem
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            candidate = Path(src_img_dir) / (img_stem + ext)
            if candidate.exists():
                img_path = candidate
                break

        if img_path and new_lines:
            shutil.copy(img_path, dst_img_dir)
            with open(os.path.join(dst_lbl_dir, txt_file.name), "w") as f:
                f.write("\n".join(new_lines) + "\n")
            added += 1

    total_added += added
    print(f"{split} -> {dst_split}: {added} images merged")

print(f"\n✅ Total no_vest images merged: {total_added}")

train -> train: 960 images merged
valid -> val: 120 images merged
⚠️ test: sirf 108 available hain, 120 nahi mil sakti
test -> test: 108 images merged

✅ Total no_vest images merged: 1188


In [8]:
yaml_content = """path: construction-ppe
train: images/train
val: images/val
test: images/test

names:
  0: helmet
  1: gloves
  2: vest
  3: goggles
  4: none
  5: Person
  6: no_helmet
  7: no_goggle
  8: no_gloves
  9: no_vest
"""
with open(f"{CONSTRUCTION_PPE_PATH}/construction-ppe.yaml", "w") as f:
    f.write(yaml_content)
print("YAML ban gayi ✅")

YAML ban gayi ✅


In [9]:
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
    return counter

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/train", names_final)
count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/val", names_final)
count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/test", names_final)


datasets/construction-ppe/labels/train:
  0 (helmet): 1357 boxes
  1 (gloves): 1162 boxes
  2 (vest): 1283 boxes
  3 (goggles): 427 boxes
  4 (none): 654 boxes
  5 (Person): 1790 boxes
  6 (no_helmet): 400 boxes
  7 (no_goggle): 337 boxes
  8 (no_gloves): 442 boxes
  9 (no_vest): 1999 boxes

datasets/construction-ppe/labels/val:
  0 (helmet): 201 boxes
  1 (gloves): 136 boxes
  2 (vest): 171 boxes
  3 (goggles): 47 boxes
  4 (none): 81 boxes
  5 (Person): 239 boxes
  6 (no_helmet): 45 boxes
  7 (no_goggle): 41 boxes
  8 (no_gloves): 56 boxes
  9 (no_vest): 238 boxes

datasets/construction-ppe/labels/test:
  0 (helmet): 192 boxes
  1 (gloves): 163 boxes
  2 (vest): 178 boxes
  3 (goggles): 52 boxes
  4 (none): 65 boxes
  5 (Person): 236 boxes
  6 (no_helmet): 40 boxes
  7 (no_goggle): 33 boxes
  8 (no_gloves): 58 boxes
  9 (no_vest): 226 boxes


Counter({2: 178,
         5: 236,
         1: 163,
         0: 192,
         3: 52,
         4: 65,
         9: 226,
         6: 40,
         7: 33,
         8: 58})

In [10]:
import os
from pathlib import Path

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"
NO_VEST_TARGET_ID = 9

train_lbl_dir = f"{CONSTRUCTION_PPE_PATH}/labels/train"
train_img_dir = f"{CONSTRUCTION_PPE_PATH}/images/train"

removed = 0
for txt_file in Path(train_lbl_dir).glob("*.txt"):
    with open(txt_file) as f:
        lines = [l.strip().split() for l in f if l.strip()]

    # Merged no_vest files mein SIRF class 9 hoti hai (kyunki source dataset mein sirf 2 classes thi)
    if lines and all(int(l[0]) == NO_VEST_TARGET_ID for l in lines):
        txt_file.unlink()  # label delete
        # matching image bhi delete karo
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = Path(train_img_dir) / (txt_file.stem + ext)
            if img_path.exists():
                img_path.unlink()
                break
        removed += 1

print(f"✅ {removed} pehle wali no_vest images train se hata di gayi")

✅ 960 pehle wali no_vest images train se hata di gayi


In [11]:
import shutil, random
from pathlib import Path

random.seed(42)

SAFETY_VEST_PATH = "/content/Safety-Vests-14"
NO_VEST_SOURCE_ID = 0
TARGET_BOXES_TRAIN = 1450   # helmet/vest ke qareeb range

src_lbl_dir = os.path.join(SAFETY_VEST_PATH, "train", "labels")
src_img_dir = os.path.join(SAFETY_VEST_PATH, "train", "images")

# sirf no-vest wali candidate files
candidates = []
for txt_file in Path(src_lbl_dir).glob("*.txt"):
    with open(txt_file) as f:
        lines = f.readlines()
    box_count = sum(1 for l in lines if l.split() and int(l.split()[0]) == NO_VEST_SOURCE_ID)
    if box_count > 0:
        candidates.append((txt_file, box_count))

random.shuffle(candidates)

selected = []
running_total = 0
for txt_file, box_count in candidates:
    if running_total >= TARGET_BOXES_TRAIN:
        break
    selected.append(txt_file)
    running_total += box_count

print(f"Selected {len(selected)} images, giving ~{running_total} no_vest boxes (target: {TARGET_BOXES_TRAIN})")

added = 0
for txt_file in selected:
    new_lines = []
    with open(txt_file) as f:
        for line in f:
            parts = line.strip().split()
            if parts and int(parts[0]) == NO_VEST_SOURCE_ID:
                parts[0] = str(NO_VEST_TARGET_ID)
                new_lines.append(" ".join(parts))

    img_stem = txt_file.stem
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = Path(src_img_dir) / (img_stem + ext)
        if candidate.exists():
            img_path = candidate
            break

    if img_path and new_lines:
        shutil.copy(img_path, train_img_dir)
        with open(os.path.join(train_lbl_dir, txt_file.name), "w") as f:
            f.write("\n".join(new_lines) + "\n")
        added += 1

print(f"✅ Train mein {added} no_vest images dobara merge hui, ~{running_total} boxes ke sath")

Selected 687 images, giving ~1450 no_vest boxes (target: 1450)
✅ Train mein 687 no_vest images dobara merge hui, ~1450 boxes ke sath


In [12]:
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
    return counter

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/train", names_final)


datasets/construction-ppe/labels/train:
  0 (helmet): 1357 boxes
  1 (gloves): 1162 boxes
  2 (vest): 1283 boxes
  3 (goggles): 427 boxes
  4 (none): 654 boxes
  5 (Person): 1790 boxes
  6 (no_helmet): 400 boxes
  7 (no_goggle): 337 boxes
  8 (no_gloves): 442 boxes
  9 (no_vest): 1450 boxes


Counter({9: 1450,
         8: 442,
         6: 400,
         7: 337,
         4: 654,
         5: 1790,
         0: 1357,
         2: 1283,
         1: 1162,
         3: 427})

In [13]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="WiZGNgO3LUcdN4N3d1MA")
project = rf.workspace("ayvu").project("ppe-v2-7zlf2")
version = project.version(2)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PPE-v2-2 in yolov8:: 100%|██████████| 7416/7416 [00:01<00:00, 4969.41it/s]


In [14]:
import yaml

PPE_V2_PATH = "/content/PPE-v2-2"

with open(f"{PPE_V2_PATH}/data.yaml") as f:
    d3_yaml = yaml.safe_load(f)

print("PPE v2 (AYVU) classes:")
for idx, name in enumerate(d3_yaml['names']):
    print(f"  {idx}: {name}")

PPE v2 (AYVU) classes:
  0: Person
  1: boots
  2: gloves
  3: goggles
  4: helmet
  5: no_boots
  6: no_gloves
  7: no_goggle
  8: no_helmet
  9: none
  10: vest


In [15]:
from pathlib import Path
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
    return counter

names_ppe_v2 = d3_yaml['names']

print("=== TRAIN ===")
count_classes(f"{PPE_V2_PATH}/train/labels", names_ppe_v2)

print("\n=== VALID ===")
count_classes(f"{PPE_V2_PATH}/valid/labels", names_ppe_v2)

print("\n=== TEST ===")
count_classes(f"{PPE_V2_PATH}/test/labels", names_ppe_v2)

=== TRAIN ===

/content/PPE-v2-2/train/labels:
  0 (Person): 5469 boxes
  1 (boots): 3702 boxes
  2 (gloves): 3421 boxes
  3 (goggles): 1251 boxes
  4 (helmet): 4181 boxes
  5 (no_boots): 258 boxes
  6 (no_gloves): 1365 boxes
  7 (no_goggle): 1050 boxes
  8 (no_helmet): 1224 boxes
  9 (none): 1977 boxes
  10 (vest): 3882 boxes

=== VALID ===

/content/PPE-v2-2/valid/labels:
  0 (Person): 192 boxes
  1 (boots): 158 boxes
  2 (gloves): 145 boxes
  3 (goggles): 50 boxes
  4 (helmet): 151 boxes
  5 (no_boots): 6 boxes
  6 (no_gloves): 43 boxes
  7 (no_goggle): 28 boxes
  8 (no_helmet): 37 boxes
  9 (none): 76 boxes
  10 (vest): 149 boxes

=== TEST ===

/content/PPE-v2-2/test/labels:
  0 (Person): 230 boxes
  1 (boots): 204 boxes
  2 (gloves): 159 boxes
  3 (goggles): 51 boxes
  4 (helmet): 189 boxes
  5 (no_boots): 23 boxes
  6 (no_gloves): 58 boxes
  7 (no_goggle): 33 boxes
  8 (no_helmet): 40 boxes
  9 (none): 62 boxes
  10 (vest): 175 boxes


Counter({10: 175,
         0: 230,
         4: 189,
         9: 62,
         5: 23,
         6: 58,
         8: 40,
         1: 204,
         3: 51,
         2: 159,
         7: 33})

In [16]:
for split in ["train", "valid", "test"]:
    img_dir = f"{PPE_V2_PATH}/{split}/images"
    lbl_dir = f"{PPE_V2_PATH}/{split}/labels"
    img_count = len(list(Path(img_dir).glob("*")))
    lbl_count = len(list(Path(lbl_dir).glob("*.txt")))
    print(f"{split}: {img_count} images, {lbl_count} label files")

train: 3429 images, 3429 label files
valid: 136 images, 136 label files
test: 137 images, 137 label files


In [17]:
import os, shutil, random
from pathlib import Path

random.seed(42)

PPE_V2_PATH = "/content/PPE-v2-2"
CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

# PPE-v2 ID -> Construction-PPE ID mapping
# PPE-v2: 0 Person, 1 boots, 2 gloves, 3 goggles, 4 helmet, 5 no_boots,
#         6 no_gloves, 7 no_goggle, 8 no_helmet, 9 none, 10 vest
# Construction-PPE (final): 0 helmet,1 gloves,2 vest,3 goggles,4 none,
#                            5 Person,6 no_helmet,7 no_goggle,8 no_gloves,9 no_vest
PPE_V2_REMAP = {
    0: 5,   # Person -> 5
    2: 1,   # gloves -> 1
    3: 3,   # goggles -> 3
    4: 0,   # helmet -> 0
    6: 8,   # no_gloves -> 8
    7: 7,   # no_goggle -> 7
    8: 6,   # no_helmet -> 6
    9: 4,   # none -> 4
    10: 2,  # vest -> 2
    # 1 (boots), 5 (no_boots) -> DROP
}

TARGET_BOXES = {
    "no_helmet": 500,   # kitne EXTRA boxes chahiye (train ke liye)
    "no_goggle": 560,
    "no_gloves": 460,
}
TARGET_CLASS_IDS_SRC = {"no_helmet": 8, "no_goggle": 7, "no_gloves": 6}  # PPE-v2 ke IDs

src_lbl_dir = f"{PPE_V2_PATH}/train/labels"
src_img_dir = f"{PPE_V2_PATH}/train/images"
dst_img_dir = f"{CONSTRUCTION_PPE_PATH}/images/train"
dst_lbl_dir = f"{CONSTRUCTION_PPE_PATH}/labels/train"

# har candidate file ke against uska contribution nikalo
candidates = []
for txt_file in Path(src_lbl_dir).glob("*.txt"):
    with open(txt_file) as f:
        lines = [l.strip().split() for l in f if l.strip()]
    contrib = Counter(int(l[0]) for l in lines)
    relevant = {name: contrib.get(cid, 0) for name, cid in TARGET_CLASS_IDS_SRC.items()}
    if sum(relevant.values()) > 0:
        candidates.append((txt_file, relevant))

random.shuffle(candidates)

running = {"no_helmet": 0, "no_goggle": 0, "no_gloves": 0}
selected = []

for txt_file, relevant in candidates:
    # agar in teeno mein se kam se kam ek abhi bhi target se kam hai, to add karo
    if any(running[k] < TARGET_BOXES[k] for k in TARGET_BOXES):
        selected.append(txt_file)
        for k in TARGET_BOXES:
            running[k] += relevant[k]
    if all(running[k] >= TARGET_BOXES[k] for k in TARGET_BOXES):
        break

print(f"Selected {len(selected)} images")
print(f"Running totals: {running}")

added = 0
for txt_file in selected:
    new_lines = []
    with open(txt_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id in PPE_V2_REMAP:
                parts[0] = str(PPE_V2_REMAP[old_id])
                new_lines.append(" ".join(parts))

    img_stem = txt_file.stem
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = Path(src_img_dir) / (img_stem + ext)
        if candidate.exists():
            img_path = candidate
            break

    if img_path and new_lines:
        # naam clash se bachne ke liye prefix lagao
        new_name = f"ppev2_{txt_file.stem}"
        shutil.copy(img_path, Path(dst_img_dir) / (new_name + img_path.suffix))
        with open(Path(dst_lbl_dir) / (new_name + ".txt"), "w") as f:
            f.write("\n".join(new_lines) + "\n")
        added += 1

print(f"\n✅ {added} images train mein merge hui")

Selected 374 images
Running totals: {'no_helmet': 628, 'no_goggle': 560, 'no_gloves': 718}

✅ 374 images train mein merge hui


In [18]:
from pathlib import Path
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    total = 0
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
        total += counter[cls_id]
    print(f"  TOTAL: {total} boxes")
    return counter

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

print("="*50)
print("TRAIN")
print("="*50)
train_counts = count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/train", names_final)

print("\n" + "="*50)
print("VALIDATION")
print("="*50)
val_counts = count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/val", names_final)

print("\n" + "="*50)
print("TEST")
print("="*50)
test_counts = count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/test", names_final)

TRAIN

datasets/construction-ppe/labels/train:
  0 (helmet): 1368 boxes
  1 (gloves): 1192 boxes
  2 (vest): 1291 boxes
  3 (goggles): 471 boxes
  4 (none): 1296 boxes
  5 (Person): 2473 boxes
  6 (no_helmet): 1028 boxes
  7 (no_goggle): 897 boxes
  8 (no_gloves): 1160 boxes
  9 (no_vest): 1450 boxes
  TOTAL: 12626 boxes

VALIDATION

datasets/construction-ppe/labels/val:
  0 (helmet): 201 boxes
  1 (gloves): 136 boxes
  2 (vest): 171 boxes
  3 (goggles): 47 boxes
  4 (none): 81 boxes
  5 (Person): 239 boxes
  6 (no_helmet): 45 boxes
  7 (no_goggle): 41 boxes
  8 (no_gloves): 56 boxes
  9 (no_vest): 238 boxes
  TOTAL: 1255 boxes

TEST

datasets/construction-ppe/labels/test:
  0 (helmet): 192 boxes
  1 (gloves): 163 boxes
  2 (vest): 178 boxes
  3 (goggles): 52 boxes
  4 (none): 65 boxes
  5 (Person): 236 boxes
  6 (no_helmet): 40 boxes
  7 (no_goggle): 33 boxes
  8 (no_gloves): 58 boxes
  9 (no_vest): 226 boxes
  TOTAL: 1243 boxes


In [19]:
print("\n" + "="*50)
print("IMAGE/LABEL FILE COUNT CHECK")
print("="*50)

for split in ["train", "val", "test"]:
    img_dir = f"{CONSTRUCTION_PPE_PATH}/images/{split}"
    lbl_dir = f"{CONSTRUCTION_PPE_PATH}/labels/{split}"
    img_count = len([f for f in Path(img_dir).glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    lbl_count = len(list(Path(lbl_dir).glob("*.txt")))
    match = "✅" if img_count == lbl_count else "⚠️ MISMATCH"
    print(f"{split}: {img_count} images, {lbl_count} labels {match}")


IMAGE/LABEL FILE COUNT CHECK
train: 2193 images, 2203 labels ⚠️ MISMATCH
val: 263 images, 263 labels ✅
test: 249 images, 249 labels ✅


In [20]:
print("\n" + "="*50)
print("SUMMARY TABLE (Train)")
print("="*50)
print(f"{'Class':<15}{'Boxes':<10}{'Status'}")
print("-"*40)
for cls_id in sorted(train_counts.keys()):
    name = names_final[cls_id]
    boxes = train_counts[cls_id]
    if boxes < 500:
        status = "🔴 Kam"
    elif boxes < 1000:
        status = "🟡 Theek"
    else:
        status = "✅ Achha"
    print(f"{name:<15}{boxes:<10}{status}")


SUMMARY TABLE (Train)
Class          Boxes     Status
----------------------------------------
helmet         1368      ✅ Achha
gloves         1192      ✅ Achha
vest           1291      ✅ Achha
goggles        471       🔴 Kam
none           1296      ✅ Achha
Person         2473      ✅ Achha
no_helmet      1028      ✅ Achha
no_goggle      897       🟡 Theek
no_gloves      1160      ✅ Achha
no_vest        1450      ✅ Achha


In [21]:
from pathlib import Path

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/train")
lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/train")

img_stems = {f.stem for f in img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
lbl_stems = {f.stem for f in lbl_dir.glob("*.txt")}

orphan_labels = lbl_stems - img_stems
orphan_images = img_stems - lbl_stems

print(f"Labels bina image ke: {len(orphan_labels)}")
for stem in list(orphan_labels)[:20]:
    print(f"  {stem}.txt")

print(f"\nImages bina label ke: {len(orphan_images)}")
for stem in list(orphan_images)[:20]:
    print(f"  {stem}")

Labels bina image ke: 10
  image945(1).txt
  image941(1).txt
  image949(1).txt
  image95(1).txt
  image950(1).txt
  image947(1).txt
  image946(1).txt
  image940(1).txt
  image948(1).txt
  image944(1).txt

Images bina label ke: 0


In [22]:
from pathlib import Path

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"
lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/train")

orphan_files = [
    "image940(1).txt", "image95(1).txt", "image949(1).txt", "image948(1).txt",
    "image947(1).txt", "image945(1).txt", "image946(1).txt", "image950(1).txt",
    "image944(1).txt", "image941(1).txt"
]

removed = 0
for fname in orphan_files:
    fpath = lbl_dir / fname
    if fpath.exists():
        fpath.unlink()
        removed += 1

print(f"✅ {removed} orphan label files delete kar diye")

# Verify
img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/train")
img_count = len([f for f in img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
lbl_count = len(list(lbl_dir.glob("*.txt")))
print(f"train: {img_count} images, {lbl_count} labels")

✅ 10 orphan label files delete kar diye
train: 2193 images, 2193 labels


In [23]:
import shutil, random
from pathlib import Path
from collections import Counter

random.seed(42)

PPE_V2_PATH = "/content/PPE-v2-2"
CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

GOGGLES_SRC_ID = 3       # PPE-v2 mein goggles ka ID
GOGGLES_TARGET_ID = 3    # Construction-PPE mein bhi goggles ka ID same hai

TARGET_EXTRA_BOXES = 500   # kitne extra boxes chahiye (473 -> ~970)

src_lbl_dir = f"{PPE_V2_PATH}/train/labels"
src_img_dir = f"{PPE_V2_PATH}/train/images"
dst_img_dir = f"{CONSTRUCTION_PPE_PATH}/images/train"
dst_lbl_dir = f"{CONSTRUCTION_PPE_PATH}/labels/train"

# candidates: jin images mein goggles hain
candidates = []
for txt_file in Path(src_lbl_dir).glob("*.txt"):
    with open(txt_file) as f:
        lines = [l.strip().split() for l in f if l.strip()]
    box_count = sum(1 for l in lines if int(l[0]) == GOGGLES_SRC_ID)
    if box_count > 0:
        candidates.append((txt_file, box_count))

random.shuffle(candidates)

# same remap jo pehle use kiya tha (PPE-v2 -> Construction-PPE IDs)
PPE_V2_REMAP = {
    0: 5,   # Person -> 5
    2: 1,   # gloves -> 1
    3: 3,   # goggles -> 3
    4: 0,   # helmet -> 0
    6: 8,   # no_gloves -> 8
    7: 7,   # no_goggle -> 7
    8: 6,   # no_helmet -> 6
    9: 4,   # none -> 4
    10: 2,  # vest -> 2
    # 1 (boots), 5 (no_boots) -> DROP
}

selected = []
running_total = 0
for txt_file, box_count in candidates:
    if running_total >= TARGET_EXTRA_BOXES:
        break
    selected.append(txt_file)
    running_total += box_count

print(f"Selected {len(selected)} images, ~{running_total} extra goggles boxes")

added = 0
for txt_file in selected:
    new_lines = []
    with open(txt_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id in PPE_V2_REMAP:
                parts[0] = str(PPE_V2_REMAP[old_id])
                new_lines.append(" ".join(parts))

    img_stem = txt_file.stem
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = Path(src_img_dir) / (img_stem + ext)
        if candidate.exists():
            img_path = candidate
            break

    if img_path and new_lines:
        new_name = f"ppev2_goggles_{txt_file.stem}"
        # duplicate check (agar yahi image pehle bhi merge ho chuki ho to skip)
        if (Path(dst_lbl_dir) / (new_name + ".txt")).exists():
            continue
        shutil.copy(img_path, Path(dst_img_dir) / (new_name + img_path.suffix))
        with open(Path(dst_lbl_dir) / (new_name + ".txt"), "w") as f:
            f.write("\n".join(new_lines) + "\n")
        added += 1

print(f"\n✅ {added} images train mein merge hui (goggles boost)")

Selected 461 images, ~500 extra goggles boxes

✅ 461 images train mein merge hui (goggles boost)


In [24]:
def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")
    return counter

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/train", names_final)

# image/label match bhi dobara check karo
img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/train")
lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/train")
img_count = len([f for f in img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
lbl_count = len(list(lbl_dir.glob("*.txt")))
print(f"\ntrain: {img_count} images, {lbl_count} labels")


datasets/construction-ppe/labels/train:
  0 (helmet): 1847 boxes
  1 (gloves): 1966 boxes
  2 (vest): 1770 boxes
  3 (goggles): 963 boxes
  4 (none): 1375 boxes
  5 (Person): 3042 boxes
  6 (no_helmet): 1070 boxes
  7 (no_goggle): 905 boxes
  8 (no_gloves): 1195 boxes
  9 (no_vest): 1450 boxes

train: 2654 images, 2654 labels


In [25]:
from pathlib import Path
from collections import Counter

def count_classes(label_dir, names):
    counter = Counter()
    for f in Path(label_dir).glob("*.txt"):
        with open(f) as file:
            for line in file:
                parts = line.strip().split()
                if parts:
                    counter[int(parts[0])] += 1
    print(f"\n{label_dir}:")
    for cls_id in sorted(counter.keys()):
        name = names[cls_id] if cls_id < len(names) else "?"
        print(f"  {cls_id} ({name}): {counter[cls_id]} boxes")

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/val", names_final)
count_classes(f"{CONSTRUCTION_PPE_PATH}/labels/test", names_final)

# image/label match check
for split in ["val", "test"]:
    img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/{split}")
    lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/{split}")
    img_count = len([f for f in img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])
    lbl_count = len(list(lbl_dir.glob("*.txt")))
    print(f"{split}: {img_count} images, {lbl_count} labels")


datasets/construction-ppe/labels/val:
  0 (helmet): 201 boxes
  1 (gloves): 136 boxes
  2 (vest): 171 boxes
  3 (goggles): 47 boxes
  4 (none): 81 boxes
  5 (Person): 239 boxes
  6 (no_helmet): 45 boxes
  7 (no_goggle): 41 boxes
  8 (no_gloves): 56 boxes
  9 (no_vest): 238 boxes

datasets/construction-ppe/labels/test:
  0 (helmet): 192 boxes
  1 (gloves): 163 boxes
  2 (vest): 178 boxes
  3 (goggles): 52 boxes
  4 (none): 65 boxes
  5 (Person): 236 boxes
  6 (no_helmet): 40 boxes
  7 (no_goggle): 33 boxes
  8 (no_gloves): 58 boxes
  9 (no_vest): 226 boxes
val: 263 images, 263 labels
test: 249 images, 249 labels


In [27]:
import shutil, random
from pathlib import Path
from collections import Counter

random.seed(42)

PPE_V2_PATH = "/content/PPE-v2-2"
CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

src_lbl_dir = f"{PPE_V2_PATH}/train/labels"
src_img_dir = f"{PPE_V2_PATH}/train/images"
dst_img_dir = f"{CONSTRUCTION_PPE_PATH}/images/train"
dst_lbl_dir = f"{CONSTRUCTION_PPE_PATH}/labels/train"

PPE_V2_REMAP = {
    0: 5,   # Person -> 5
    2: 1,   # gloves -> 1
    3: 3,   # goggles -> 3
    4: 0,   # helmet -> 0
    6: 8,   # no_gloves -> 8
    7: 7,   # no_goggle -> 7
    8: 6,   # no_helmet -> 6
    9: 4,   # none -> 4
    10: 2,  # vest -> 2
    # 1 (boots), 5 (no_boots) -> DROP
}

# ---- Step 1: already-used PPE-v2 stems track karo (dono purane merges se) ----
used_stems = set()
for f in Path(dst_lbl_dir).glob("ppev2_*.txt"):
    stem = f.stem
    if stem.startswith("ppev2_goggles_"):
        orig = stem.replace("ppev2_goggles_", "")
    else:
        orig = stem.replace("ppev2_", "")
    used_stems.add(orig)

print(f"Pehle se {len(used_stems)} PPE-v2 images use ho chuki hain (skip karenge)")

# ---- Step 2: targets ----
TARGET_BOXES = {
    "goggles": 350,     # 963 -> ~1313
    "no_helmet": 250,   # 1070 -> ~1320
    "no_goggle": 350,   # 905 -> ~1255
}
TARGET_CLASS_IDS_SRC = {"goggles": 3, "no_helmet": 8, "no_goggle": 7}  # PPE-v2 IDs

# ---- Step 3: fresh candidates (used_stems exclude karke) ----
candidates = []
for txt_file in Path(src_lbl_dir).glob("*.txt"):
    if txt_file.stem in used_stems:
        continue
    with open(txt_file) as f:
        lines = [l.strip().split() for l in f if l.strip()]
    contrib = Counter(int(l[0]) for l in lines)
    relevant = {name: contrib.get(cid, 0) for name, cid in TARGET_CLASS_IDS_SRC.items()}
    if sum(relevant.values()) > 0:
        candidates.append((txt_file, relevant))

random.shuffle(candidates)

running = {k: 0 for k in TARGET_BOXES}
selected = []

for txt_file, relevant in candidates:
    if any(running[k] < TARGET_BOXES[k] for k in TARGET_BOXES):
        selected.append(txt_file)
        for k in TARGET_BOXES:
            running[k] += relevant[k]
    if all(running[k] >= TARGET_BOXES[k] for k in TARGET_BOXES):
        break

print(f"Selected {len(selected)} images")
print(f"Running totals: {running}")

if len(selected) == 0 or any(running[k] < TARGET_BOXES[k] for k in TARGET_BOXES):
    print("⚠️ PPE-v2 mein enough fresh images nahi mili — target puri nahi hui. Doosra source dataset chahiye hoga.")

# ---- Step 4: copy + remap + save ----
added = 0
for txt_file in selected:
    new_lines = []
    with open(txt_file) as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            old_id = int(parts[0])
            if old_id in PPE_V2_REMAP:
                parts[0] = str(PPE_V2_REMAP[old_id])
                new_lines.append(" ".join(parts))

    img_stem = txt_file.stem
    img_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        candidate = Path(src_img_dir) / (img_stem + ext)
        if candidate.exists():
            img_path = candidate
            break

    if img_path and new_lines:
        new_name = f"ppev2_r3_{txt_file.stem}"   # round-3 prefix, unique
        if (Path(dst_lbl_dir) / (new_name + ".txt")).exists():
            continue
        shutil.copy(img_path, Path(dst_img_dir) / (new_name + img_path.suffix))
        with open(Path(dst_lbl_dir) / (new_name + ".txt"), "w") as f:
            f.write("\n".join(new_lines) + "\n")
        added += 1

print(f"\n✅ {added} images train mein merge hui (round 3: goggles + no_helmet + no_goggle boost)")

Pehle se 825 PPE-v2 images use ho chuki hain (skip karenge)
Selected 704 images
Running totals: {'goggles': 518, 'no_helmet': 415, 'no_goggle': 353}

✅ 704 images train mein merge hui (round 3: goggles + no_helmet + no_goggle boost)


In [29]:
from pathlib import Path
from collections import Counter
import random

random.seed(42)

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"
lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/train")
img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/train")

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

# Upper bound jo hum chahte hain (range ka top)
UPPER_BOUND = 1500
# Protected classes — inka count kabhi is se neeche na jaye
PROTECTED_MIN = {
    0: 1200,  # helmet
    1: 1200,  # gloves
    2: 1200,  # vest
    3: 1200,  # goggles
    4: 1200,  # none
    6: 1200,  # no_helmet
    7: 1200,  # no_goggle
    8: 1200,  # no_gloves
    9: 1200,  # no_vest
    # Person (5) protected nahi — usko bhi trim karenge kyunki bohot zyada hai
}

# Step 1: sab labels parse karo, per-image class counts nikalo
image_data = {}
global_counts = Counter()
for f in lbl_dir.glob("*.txt"):
    with open(f) as file:
        lines = [l.strip().split() for l in file if l.strip()]
    counts = Counter(int(l[0]) for l in lines)
    image_data[f.stem] = counts
    global_counts.update(counts)

print("Current totals:", dict(sorted(global_counts.items())))

# Step 2: over-target classes identify karo
over_target = {c for c in global_counts if global_counts[c] > UPPER_BOUND}
print(f"\nOver target classes: {[names_final[c] for c in over_target]}")

# Step 3: candidates for removal = images jinme SIRF over-target classes hain
# (koi bhi under-target/protected class na ho)
removal_candidates = []
for stem, counts in image_data.items():
    classes_in_img = set(counts.keys())
    # agar is image mein koi bhi protected/under-target class hai, to skip (safe rakho)
    if classes_in_img - over_target:  # kuch aisi class hai jo over-target nahi
        continue
    removal_candidates.append(stem)

random.shuffle(removal_candidates)
print(f"\nRemoval candidates (pure over-target images): {len(removal_candidates)}")

# Step 4: iteratively remove karo jab tak koi over-target class UPPER_BOUND se upar hai
running_counts = Counter(global_counts)
to_remove = []

for stem in removal_candidates:
    # check karo agar is image ko hata dein to koi protected class apne min se neeche to nahi jayegi
    counts = image_data[stem]
    safe = True
    for cls, cnt in counts.items():
        if cls in PROTECTED_MIN:
            if running_counts[cls] - cnt < PROTECTED_MIN[cls]:
                safe = False
                break
    if not safe:
        continue

    # sirf tab remove karo jab kam se kam ek over-target class abhi bhi bound se upar hai
    still_over = any(running_counts[c] > UPPER_BOUND for c in over_target)
    if not still_over:
        break

    to_remove.append(stem)
    running_counts.subtract(counts)

print(f"\n{len(to_remove)} images remove karne ke liye select hui")
print("Projected new totals:", dict(sorted(running_counts.items())))

Current totals: {0: 2411, 1: 2851, 2: 2343, 3: 1481, 4: 1799, 5: 4055, 6: 1485, 7: 1258, 8: 1641, 9: 1450}

Over target classes: ['helmet', 'gloves', 'vest', 'none', 'Person', 'no_gloves']

Removal candidates (pure over-target images): 530

530 images remove karne ke liye select hui
Projected new totals: {0: 1500, 1: 2400, 2: 1503, 3: 1481, 4: 1569, 5: 3154, 6: 1485, 7: 1258, 8: 1639, 9: 1450}


In [30]:
from pathlib import Path
from collections import Counter
import random

random.seed(42)

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"
lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/train")
img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/train")

names_final = ["helmet","gloves","vest","goggles","none","Person",
               "no_helmet","no_goggle","no_gloves","no_vest"]

UPPER_BOUND = 1500
PROTECTED_MIN = {
    0: 1200, 1: 1200, 2: 1200, 3: 1200, 4: 1200,
    6: 1200, 7: 1200, 8: 1200, 9: 1200,
}

# Step 1: parse
image_data = {}
global_counts = Counter()
for f in lbl_dir.glob("*.txt"):
    with open(f) as file:
        lines = [l.strip().split() for l in file if l.strip()]
    counts = Counter(int(l[0]) for l in lines)
    image_data[f.stem] = counts
    global_counts.update(counts)

over_target = {c for c in global_counts if global_counts[c] > UPPER_BOUND}

removal_candidates = []
for stem, counts in image_data.items():
    classes_in_img = set(counts.keys())
    if classes_in_img - over_target:
        continue
    removal_candidates.append(stem)

random.shuffle(removal_candidates)

running_counts = Counter(global_counts)
to_remove = []

for stem in removal_candidates:
    counts = image_data[stem]
    safe = True
    for cls, cnt in counts.items():
        if cls in PROTECTED_MIN:
            if running_counts[cls] - cnt < PROTECTED_MIN[cls]:
                safe = False
                break
    if not safe:
        continue

    still_over = any(running_counts[c] > UPPER_BOUND for c in over_target)
    if not still_over:
        break

    to_remove.append(stem)
    running_counts.subtract(counts)

print(f"{len(to_remove)} images delete honge")
print("Projected totals after delete:", dict(sorted(running_counts.items())))

# ---- Step 2: ACTUAL DELETE ----
confirm = input(f"\n⚠️  {len(to_remove)} images + labels PERMANENTLY delete karni hain. Type 'YES' to confirm: ")

if confirm.strip() == "YES":
    deleted = 0
    for stem in to_remove:
        # label delete
        lbl_path = lbl_dir / f"{stem}.txt"
        if lbl_path.exists():
            lbl_path.unlink()

        # image delete (extension pata nahi to sab try karo)
        found = False
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = img_dir / f"{stem}{ext}"
            if img_path.exists():
                img_path.unlink()
                found = True
                break
        if found:
            deleted += 1

    print(f"\n✅ {deleted} images + labels delete ho gaye")
else:
    print("\n❌ Cancelled — kuch delete nahi hua")

530 images delete honge
Projected totals after delete: {0: 1500, 1: 2400, 2: 1503, 3: 1481, 4: 1569, 5: 3154, 6: 1485, 7: 1258, 8: 1639, 9: 1450}

⚠️  530 images + labels PERMANENTLY delete karni hain. Type 'YES' to confirm: YES

✅ 530 images + labels delete ho gaye


In [31]:
import os
from pathlib import Path
from collections import Counter
import yaml

CONSTRUCTION_PPE_PATH = "datasets/construction-ppe"

# ===== Load YAML =====
with open(f"{CONSTRUCTION_PPE_PATH}/construction-ppe.yaml") as f:
    data_yaml = yaml.safe_load(f)

NUM_CLASSES = len(data_yaml['names'])
CLASS_NAMES = data_yaml['names']

print("="*60)
print(f"YAML Classes ({NUM_CLASSES}):", CLASS_NAMES)
print("="*60)

def check_split(split):
    print(f"\n{'='*60}")
    print(f"CHECKING: {split.upper()}")
    print(f"{'='*60}")

    img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/{split}")
    lbl_dir = Path(f"{CONSTRUCTION_PPE_PATH}/labels/{split}")

    img_files = {f.stem: f for f in img_dir.glob("*") if f.suffix.lower() in [".jpg", ".jpeg", ".png"]}
    lbl_files = {f.stem: f for f in lbl_dir.glob("*.txt")}

    issues = {
        "missing_labels": [],      # image hai, label nahi
        "missing_images": [],      # label hai, image nahi
        "empty_labels": [],        # label file khaali hai
        "invalid_class_id": [],    # class ID range se bahar
        "invalid_bbox_count": [],  # line mein 5 values nahi hain
        "invalid_bbox_range": [],  # coordinates 0-1 range se bahar
        "corrupt_image": [],       # image khulti nahi
        "zero_size_image": [],     # 0 KB image
    }

    # 1. Missing pairs check
    issues["missing_labels"] = list(set(img_files.keys()) - set(lbl_files.keys()))
    issues["missing_images"] = list(set(lbl_files.keys()) - set(img_files.keys()))

    # 2. Corrupt / zero-size image check
    from PIL import Image
    for stem, img_path in img_files.items():
        try:
            if img_path.stat().st_size == 0:
                issues["zero_size_image"].append(str(img_path))
                continue
            with Image.open(img_path) as im:
                im.verify()
        except Exception:
            issues["corrupt_image"].append(str(img_path))

    # 3. Label content validation
    class_counter = Counter()
    total_boxes = 0
    for stem, lbl_path in lbl_files.items():
        with open(lbl_path) as f:
            lines = [l.strip() for l in f if l.strip()]

        if len(lines) == 0:
            issues["empty_labels"].append(str(lbl_path))
            continue

        for line in lines:
            parts = line.split()
            if len(parts) != 5:
                issues["invalid_bbox_count"].append(f"{lbl_path}: '{line}'")
                continue

            cls_id = int(parts[0])
            if cls_id < 0 or cls_id >= NUM_CLASSES:
                issues["invalid_class_id"].append(f"{lbl_path}: class_id={cls_id}")
                continue

            coords = [float(p) for p in parts[1:]]
            if any(c < 0 or c > 1 for c in coords):
                issues["invalid_bbox_range"].append(f"{lbl_path}: {coords}")
                continue

            class_counter[cls_id] += 1
            total_boxes += 1

    # ===== REPORT =====
    print(f"\nTotal images: {len(img_files)}")
    print(f"Total labels: {len(lbl_files)}")
    print(f"Total valid boxes: {total_boxes}")

    print("\nClass distribution:")
    for cls_id in range(NUM_CLASSES):
        count = class_counter.get(cls_id, 0)
        flag = "🔴" if count < 50 else "✅"
        print(f"  {flag} {cls_id} ({CLASS_NAMES[cls_id]}): {count}")

    print("\n--- ISSUES ---")
    any_issue = False
    for issue_type, items in issues.items():
        if items:
            any_issue = True
            print(f"⚠️ {issue_type}: {len(items)} found")
            for item in items[:5]:
                print(f"    {item}")
            if len(items) > 5:
                print(f"    ... aur {len(items)-5} zyada")

    if not any_issue:
        print("✅ Koi issue nahi mila — split clean hai")

    return issues, class_counter

# ===== Run for all splits =====
all_issues = {}
for split in ["train", "val", "test"]:
    issues, counts = check_split(split)
    all_issues[split] = issues

# ===== Duplicate image check (across splits — data leakage) =====
print(f"\n{'='*60}")
print("DATA LEAKAGE CHECK (duplicate images across splits)")
print(f"{'='*60}")

import hashlib

def file_hash(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

split_hashes = {}
for split in ["train", "val", "test"]:
    img_dir = Path(f"{CONSTRUCTION_PPE_PATH}/images/{split}")
    hashes = set()
    for img_path in img_dir.glob("*"):
        if img_path.suffix.lower() in [".jpg", ".jpeg", ".png"]:
            hashes.add(file_hash(img_path))
    split_hashes[split] = hashes

overlap_train_val = split_hashes["train"] & split_hashes["val"]
overlap_train_test = split_hashes["train"] & split_hashes["test"]
overlap_val_test = split_hashes["val"] & split_hashes["test"]

print(f"Train-Val overlap: {len(overlap_train_val)} duplicate images")
print(f"Train-Test overlap: {len(overlap_train_test)} duplicate images")
print(f"Val-Test overlap: {len(overlap_val_test)} duplicate images")

if overlap_train_val or overlap_train_test or overlap_val_test:
    print("⚠️ WARNING: Data leakage detected — same image train aur val/test dono mein hai!")
else:
    print("✅ Koi data leakage nahi — sab splits clean separate hain")

print(f"\n{'='*60}")
print("FINAL VERDICT")
print(f"{'='*60}")
total_issues = sum(len(v) for split_issues in all_issues.values() for v in split_issues.values())
if total_issues == 0 and not (overlap_train_val or overlap_train_test or overlap_val_test):
    print("✅✅✅ DATASET COMPLETELY CLEAN — TRAINING KE LIYE READY HAI")
else:
    print(f"⚠️ {total_issues} issues mile — upar dekh ke fix karo training se pehle")

YAML Classes (10): {0: 'helmet', 1: 'gloves', 2: 'vest', 3: 'goggles', 4: 'none', 5: 'Person', 6: 'no_helmet', 7: 'no_goggle', 8: 'no_gloves', 9: 'no_vest'}

CHECKING: TRAIN

Total images: 2828
Total labels: 2828
Total valid boxes: 17439

Class distribution:
  ✅ 0 (helmet): 1500
  ✅ 1 (gloves): 2400
  ✅ 2 (vest): 1503
  ✅ 3 (goggles): 1481
  ✅ 4 (none): 1569
  ✅ 5 (Person): 3154
  ✅ 6 (no_helmet): 1485
  ✅ 7 (no_goggle): 1258
  ✅ 8 (no_gloves): 1639
  ✅ 9 (no_vest): 1450

--- ISSUES ---
✅ Koi issue nahi mila — split clean hai

CHECKING: VAL

Total images: 263
Total labels: 263
Total valid boxes: 1255

Class distribution:
  ✅ 0 (helmet): 201
  ✅ 1 (gloves): 136
  ✅ 2 (vest): 171
  🔴 3 (goggles): 47
  ✅ 4 (none): 81
  ✅ 5 (Person): 239
  🔴 6 (no_helmet): 45
  🔴 7 (no_goggle): 41
  ✅ 8 (no_gloves): 56
  ✅ 9 (no_vest): 238

--- ISSUES ---
✅ Koi issue nahi mila — split clean hai

CHECKING: TEST

Total images: 249
Total labels: 249
Total valid boxes: 1243

Class distribution:
  ✅ 0 (helmet):

In [32]:
print("=== construction-ppe.yaml ===")
with open("datasets/construction-ppe/construction-ppe.yaml") as f:
    print(f.read())

print("\n=== data.yaml ===")
with open("datasets/construction-ppe/data.yaml") as f:
    print(f.read())

=== construction-ppe.yaml ===
path: construction-ppe
train: images/train
val: images/val
test: images/test

names:
  0: helmet
  1: gloves
  2: vest
  3: goggles
  4: none
  5: Person
  6: no_helmet
  7: no_goggle
  8: no_gloves
  9: no_vest


=== data.yaml ===
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# Construction-PPE dataset by Ultralytics
# Documentation: https://docs.ultralytics.com/datasets/detect/construction-ppe/
# Example usage: yolo train data=construction-ppe.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── construction-ppe ← downloads here (178.4 MB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: construction-ppe # dataset root dir
train: images/train # train images (relative to 'path') 1132 images
val: images/val # val images (relative to 'path') 143 images
test: images/test # test images (relative to 'path') 141 images

# Classes
names:
  0: helmet
  1: glov

In [33]:
from pathlib import Path
import yaml

yaml_path = "datasets/construction-ppe/construction-ppe.yaml"

with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

base_dir = Path(yaml_path).parent
data_root = base_dir / cfg.get("path", ".")

print(f"Resolved data root: {data_root.resolve()}")

for split in ["train", "val", "test"]:
    split_path = data_root / cfg[split]
    imgs = list(split_path.glob("*"))
    status = "✅" if len(imgs) > 0 else "❌"
    print(f"{status} {split}: {split_path} -> {len(imgs)} files")

Resolved data root: /content/datasets/construction-ppe
✅ train: datasets/construction-ppe/images/train -> 2828 files
✅ val: datasets/construction-ppe/images/val -> 263 files
✅ test: datasets/construction-ppe/images/test -> 249 files


In [36]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data="datasets/construction-ppe/construction-ppe.yaml",
    epochs=60,
    imgsz=640,
    batch=16,
    patience=10,
    cache=True,
    amp=True,
    workers=8,
    device=0,
    project="runs/train",
    name="construction_ppe_final",

    # class imbalance handling
    cls=0.7,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    close_mosaic=10,

    # training stability
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,

    # monitoring / reproducibility
    seed=42,
    save_period=10,
    plots=True,
    val=True,
)

Ultralytics 8.4.112 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=True, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.7, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=datasets/construction-ppe/construction-ppe.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=construction_p

In [37]:
from google.colab import files
files.download('/content/runs/detect/runs/train/construction_ppe_final-2/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [38]:
import shutil
from google.colab import files

shutil.make_archive('construction_ppe_results', 'zip',
                     '/content/runs/detect/runs/train/construction_ppe_final-2')
files.download('construction_ppe_results.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>